## This notebook
Calculates the percentage of segdup in the genome 

In [1]:
import numpy as np
from scipy.stats import fisher_exact, binomtest, hypergeom
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev
import pyranges as pr

In [2]:
# New-annotation GFF (final AGAT-normalized annotation; re-run with this path)
GFF_PATH = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/annotation-merging/output/hifiasm-041425-denovoEnhanced_peaks2utr_sorted.agat.gff3"
gff = pd.read_csv(GFF_PATH, sep='\t', comment="#", header=None,
                  names=["Sequence","source","type","Gene Start","Gene End","score","Strand","phase","attributes"])

gff = gff[gff["type"]=="gene"].copy()
gff['gene'] = gff['attributes'].str.extract(r'ID=gene-([^;]+)', expand=False).str.upper()

In [3]:
# Segmental-duplication BEDPE (genome-level, unchanged)
BEDPE_PATH = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_duplicateLinkRemoved.bedpe"
dup = pd.read_csv(BEDPE_PATH, sep="\t", header=None,
                  names=["chr1","start1","end1","chr2","start2","end2","reference","score","strand1","strand2","max_len","aln_len","cigar","optional"])

In [4]:
def merge_alignments(df, chrom_col='chr1', start_col='start1', end_col='end1', type_col='aln_type'):
    """
    Merge overlapping or adjacent alignments of the same type on the same chromosome
    Returns a DataFrame with merged intervals and their total lengths
    """
    merged_data = []
    
    # Group by chromosome and alignment type
    for (chrom, aln_type), group in df.groupby([chrom_col, type_col]):
        # Sort by start position
        sorted_group = group.sort_values(start_col)
        
        merged_intervals = []
        current_interval = None
        
        for _, row in sorted_group.iterrows():
            start, end = row[start_col], row[end_col]
            
            if current_interval is None:
                # Start first interval
                current_interval = [start, end]
            elif start <= current_interval[1]:
                # Overlapping or adjacent - merge
                current_interval[1] = max(current_interval[1], end)
            else:
                # Non-overlapping - save current and start new
                merged_intervals.append({
                    'chrom': chrom,
                    'aln_type': aln_type,
                    'start': current_interval[0],
                    'end': current_interval[1],
                    'length': current_interval[1] - current_interval[0]
                })
                current_interval = [start, end]
        
        # Don't forget the last interval
        if current_interval is not None:
            merged_intervals.append({
                'chrom': chrom,
                'aln_type': aln_type,
                'start': current_interval[0],
                'end': current_interval[1],
                'length': current_interval[1] - current_interval[0]
            })
        
        merged_data.extend(merged_intervals)
    
    return pd.DataFrame(merged_data)

In [5]:
dup["aln_type"]="all"

In [6]:
merged_dup = merge_alignments(dup)

In [7]:
import numpy as np
from intervaltree import IntervalTree

# Build per-chromosome interval trees from BOTH sides of each BEDPE pair (vectorized)
c1 = dup["chr1"].values; s1 = dup["start1"].astype(int).values; e1 = dup["end1"].astype(int).values
c2 = dup["chr2"].values; s2 = dup["start2"].astype(int).values; e2 = dup["end2"].astype(int).values
chroms = np.concatenate([c1, c2]); starts = np.concatenate([s1, s2]); ends = np.concatenate([e1, e2])

trees = {}
for chrom in np.unique(chroms):
    m = chroms == chrom
    trees[chrom] = IntervalTree.from_tuples(zip(starts[m], ends[m], [True] * int(m.sum())))

# Gene intervals (GFF 1-based inclusive -> 0-based half-open)
all_features = gff.copy()
all_features["Start"] = all_features["Gene Start"].astype(int) - 1
all_features["End"] = all_features["Gene End"].astype(int)
all_features = all_features.rename(columns={"Sequence": "Chromosome"})

# overlaps_dup = gene interval overlaps any segdup interval (either side)
def overlaps_dup(row):
    tree = trees.get(row["Chromosome"])
    return bool(tree.overlaps(int(row["Start"]), int(row["End"]))) if tree is not None else False

all_features["overlaps_dup"] = all_features.apply(overlaps_dup, axis=1)

print("overlaps_dup value_counts:")
print(all_features["overlaps_dup"].value_counts().to_string())


overlaps_dup value_counts:
overlaps_dup
False    15876
True     10397


In [9]:
all_features.to_csv("./hifiasm_gene_segDup_overlapInfo_081726.tsv", sep="\t")